# Mail Guard AI — Model Training

This notebook fine-tunes **DistilBERT** on a multi-source spam dataset and exports the model in both PyTorch and ONNX formats for production inference.

**Runtime:** Set to GPU via `Runtime > Change runtime type > T4 GPU`

**Time:** ~30 minutes on a T4 GPU

---

## 1. Setup and Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn optimum[onnxruntime] shap

In [ ]:
import os
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

print("All imports successful.")

In [ ]:
# Verify GPU is available
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be much slower (~2-4 hours).")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 2. Dataset Loading

We load the SMS Spam Collection dataset from Kaggle. If you have additional datasets (Enron, SpamAssassin), upload them and uncomment the relevant sections.

In [ ]:
# Upload your spam.csv file using the file browser on the left,
# or download it directly:
!pip install -q kaggle

# Option 1: Upload spam.csv manually via Colab file browser (left sidebar)
# Option 2: Download from a direct URL
# For now, we'll use the HuggingFace datasets library as a reliable source

from datasets import load_dataset

# Load SMS Spam Collection from HuggingFace
try:
    hf_dataset = load_dataset("ucirvine/sms_spam", trust_remote_code=True)
    sms_df = hf_dataset["train"].to_pandas()
    sms_df.columns = ["label", "text"] if "sms" not in sms_df.columns else sms_df.columns
    # Map labels
    if sms_df["label"].dtype == "object":
        sms_df["label"] = sms_df["label"].map({"ham": 0, "spam": 1})
    sms_df["source"] = "sms_spam_collection"
    print(f"SMS Spam Collection: {len(sms_df)} messages")
    print(f"  Spam: {sms_df['label'].sum()} | Ham: {(sms_df['label'] == 0).sum()}")
except Exception as e:
    print(f"HuggingFace load failed: {e}")
    print("Falling back to manual upload...")
    # Upload spam.csv to Colab and uncomment:
    # sms_df = pd.read_csv('spam.csv', encoding='latin-1')[['v1', 'v2']]
    # sms_df.columns = ['label', 'text']
    # sms_df['label'] = sms_df['label'].map({'ham': 0, 'spam': 1})
    # sms_df['source'] = 'sms_spam_collection'
    raise

In [ ]:
# --- Optional: Load Enron Spam Corpus ---
# If you have the Enron dataset, upload and uncomment:
#
# enron_records = []
# for label_name, label_int in [("spam", 1), ("ham", 0)]:
#     label_dir = Path(f"enron/{label_name}")
#     if label_dir.exists():
#         for f in label_dir.glob("*.txt"):
#             text = f.read_text(encoding="utf-8", errors="ignore").strip()
#             if len(text) > 10:
#                 enron_records.append({"text": text, "label": label_int})
# enron_df = pd.DataFrame(enron_records)
# enron_df["source"] = "enron"
# print(f"Enron: {len(enron_df)} emails")

print("Enron dataset: skipped (not uploaded). SMS dataset will be used.")

## 3. Data Preprocessing

In [ ]:
# Merge all available datasets
datasets_list = [sms_df]
# Uncomment if you loaded Enron:
# datasets_list.append(enron_df)

df = pd.concat(datasets_list, ignore_index=True)
print(f"Total dataset: {len(df)} samples")

# Clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = " ".join(text.split())  # Normalize whitespace
    if len(text) > 5000:
        text = text[:5000]
    return text.strip()

df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.len() > 10]
print(f"After cleaning: {len(df)} samples")

# Deduplicate
df["hash"] = df["text"].apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
before = len(df)
df = df.drop_duplicates(subset=["hash"]).drop(columns=["hash"])
print(f"After dedup: {len(df)} samples (removed {before - len(df)} duplicates)")

# Display class distribution
print(f"\nClass distribution:")
print(f"  Ham:  {(df['label'] == 0).sum()} ({(df['label'] == 0).mean()*100:.1f}%)")
print(f"  Spam: {(df['label'] == 1).sum()} ({(df['label'] == 1).mean()*100:.1f}%)")

In [ ]:
# Create stratified train/val/test splits
train_val, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1176, stratify=train_val["label"], random_state=42
    # 0.1176 of 0.85 ≈ 0.10 of total
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

## 4. Tokenization

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize, batched=True, batch_size=64)
val_dataset = val_dataset.map(tokenize, batched=True, batch_size=64)
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=64)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenization complete.")
print(f"  Sample input_ids shape: {train_dataset[0]['input_ids'].shape}")

## 5. Model Training

In [ ]:
# Load pre-trained DistilBERT with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "ham", 1: "spam"},
    label2id={"ham": 0, "spam": 1},
)

print(f"Model parameters: {model.num_parameters():,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
    }

OUTPUT_DIR = "./distilbert-spam-v2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete in {train_result.metrics['train_runtime']:.0f} seconds.")

## 6. Evaluation on Test Set

In [ ]:
# Evaluate on held-out test set
test_results = trainer.evaluate(test_dataset)

print("Test Set Results")
print("=" * 40)
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")

In [ ]:
# Generate predictions for confusion matrix
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Classification report
print("Classification Report")
print("=" * 50)
print(classification_report(y_true, y_pred, target_names=["Ham", "Spam"]))

In [ ]:
# Confusion Matrix visualization
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"],
    ax=ax,
    annot_kws={"size": 16},
)
ax.set_xlabel("Predicted", fontsize=14)
ax.set_ylabel("Actual", fontsize=14)
ax.set_title("Mail Guard AI — Confusion Matrix (Test Set)", fontsize=16)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# Training history visualization
log_history = trainer.state.log_history

# Extract epoch-level metrics
eval_logs = [log for log in log_history if "eval_f1" in log]
epochs = [log["epoch"] for log in eval_logs]
f1_scores = [log["eval_f1"] for log in eval_logs]
acc_scores = [log["eval_accuracy"] for log in eval_logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, f1_scores, "o-", color="#2563eb", linewidth=2, markersize=8)
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("F1 Score", fontsize=12)
ax1.set_title("F1 Score per Epoch", fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.9, 1.0)

ax2.plot(epochs, acc_scores, "o-", color="#059669", linewidth=2, markersize=8)
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Accuracy per Epoch", fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.9, 1.0)

plt.suptitle("Mail Guard AI — Training History", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_history.png")

## 7. Save Model

In [ ]:
# Save the fine-tuned model and tokenizer
FINAL_MODEL_DIR = "./distilbert-spam-v2-final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"Model saved to {FINAL_MODEL_DIR}")
print(f"Contents:")
for f in sorted(Path(FINAL_MODEL_DIR).iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

## 8. ONNX Export

Export to ONNX Runtime format for 3-5x faster CPU inference in production.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

# Export to ONNX
ort_model = ORTModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_DIR, export=True
)
ort_model.save_pretrained(FINAL_MODEL_DIR)

print(f"ONNX model exported to {FINAL_MODEL_DIR}")
onnx_path = Path(FINAL_MODEL_DIR) / "model.onnx"
if onnx_path.exists():
    print(f"  model.onnx size: {onnx_path.stat().st_size / 1e6:.1f} MB")

## 9. Quick Inference Test

In [ ]:
import time

# Load the saved model for inference test
test_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
test_model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
test_model.eval()

test_emails = [
    ("CONGRATULATIONS! You have been selected to WIN $1,000,000! Click here NOW to claim your FREE prize! Act IMMEDIATELY!", "spam"),
    ("Hi Sarah, just following up on the quarterly report. Can you send the updated figures by Friday? Thanks, John", "ham"),
    ("URGENT: Your account has been compromised. Verify your identity immediately at http://bit.ly/verify-now or your account will be suspended.", "spam"),
    ("Hey, are we still on for lunch tomorrow? I was thinking the Italian place on 5th street.", "ham"),
    ("Dear Customer, you have won a FREE iPhone 15! Reply with your credit card number to claim. Limited time offer!!!", "spam"),
    ("The meeting has been rescheduled to 3pm on Thursday. Please update your calendar accordingly.", "ham"),
    ("Make $$$ from home! No experience needed! Earn $5000/week with this simple trick! Visit www.easy-money-now.com", "spam"),
    ("Please find attached the invoice for last month. Let me know if you have any questions about the charges.", "ham"),
]

print("Inference Test Results")
print("=" * 70)
correct = 0

for text, expected in test_emails:
    start = time.perf_counter()
    inputs = test_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = test_model(**inputs)
    elapsed_ms = (time.perf_counter() - start) * 1000

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_label = "spam" if probs[1] > probs[0] else "ham"
    confidence = probs[1].item() if pred_label == "spam" else probs[0].item()
    status = "PASS" if pred_label == expected else "FAIL"
    correct += 1 if pred_label == expected else 0

    print(f"  [{status}] {pred_label:4s} ({confidence:.2%}) | {text[:70]}...")
    print(f"         Expected: {expected} | Latency: {elapsed_ms:.0f}ms")
    print()

print(f"Result: {correct}/{len(test_emails)} correct")

## 10. Download Model

Download the trained model to your local machine, then place it in:
```
mail-guard-api/packages/ml-service/models/distilbert-spam-v2/
```

In [ ]:
# Zip the model directory for download
import shutil

zip_path = shutil.make_archive("distilbert-spam-v2", "zip", ".", FINAL_MODEL_DIR)
print(f"Model archive created: {zip_path}")
print(f"Size: {Path(zip_path).stat().st_size / 1e6:.1f} MB")
print()
print("Download this file and extract it to:")
print("  packages/ml-service/models/distilbert-spam-v2/")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — download the zip file manually.")

---

## Summary

| Metric | Value |
|--------|-------|
| Base Model | distilbert-base-uncased |
| Training Epochs | 3 |
| Dataset Size | ~5,500 (SMS) |
| Test F1 | See results above |
| Inference Format | PyTorch + ONNX |

**Next step:** Place the downloaded model in `packages/ml-service/models/distilbert-spam-v2/` and start the ML service.